# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a [Croissant](https://mlcommons.org/croissant/) schema at the URL above.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All record sets, fields, and columns are referenced by their `@id`.

In [ ]:
# List all record sets and their structure using @id
from pprint import pprint

print("Record sets available in the dataset (@id):")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"- Record set: {rs['@id']} (name: {rs['name']})")
    print("  Fields and columns:")
    for field in rs['fields']:
        col_ids = field.get('columns', [])
        col_str = ', '.join([c['@id'] if isinstance(c, dict) else c for c in col_ids])
        print(f"    - Field: {field['@id']} (name: {field.get('name', '?')}) -> Columns: [{col_str}]")
    print("")

You can explore an example of records from each record set below. Replace `record_set_id` with the desired record set `@id`.

In [ ]:
# Example: Preview a few records from a record set by @id

# For this dataset, the main record set is usually the table: proteins, abundance, etc.
# Let's display the record set IDs again to help users pick one.

record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Record Set IDs: {record_set_ids}")

selected_record_set_id = record_set_ids[0] if record_set_ids else None
if selected_record_set_id:
    print(f"\nExample records from {selected_record_set_id}:")
    for idx, record in enumerate(dataset.records(record_set=selected_record_set_id)):
        pprint(record)
        if idx >= 2:
            break
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references are based on record set `@id`.

In [ ]:
# For this dataset, extract all main tables by @id
dataframes = {}
for rs in record_sets:
    rid = rs['@id']
    print(f"Loading data for record set {rid} ...")
    records = list(dataset.records(record_set=rid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"Columns for record set {rid}: {df.columns.tolist()[:8]}{'...' if len(df.columns)>8 else ''}")
        print(df.head(2))
    else:
        print(f"No records found for {rid}.")
    print("")
# Pick the first loaded table as reference for further steps
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain analysis will proceed with record set: {main_record_set_id}")
    print(f"Sample columns: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Here, we explore the loaded data. We demonstrate filtering based on numeric values, normalization, and grouping. Please ensure to use column names (derived from column `@id`s) as printed above.

👉 **Tip:** If you are unsure about which columns to analyze, refer to the previous cell for the column names (by `@id`).

In [ ]:
# Choose a numeric field (e.g., abundance, MW, coverage, etc. by column @id)
# We'll attempt to pick a suitable numeric column if available.
df = dataframes[main_record_set_id]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    # Try infer columns that look numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns available: {numeric_cols}")
if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Proceeding with numeric field (column @id): {numeric_field}")

    # Filtering: keep records where value > threshold
    threshold = df[numeric_field].mean()  # Use mean as a threshold example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}: {len(filtered_df)} records out of {len(df)}")

    # Normalization
    filtered_df[f"{numeric_field}_zscore"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_zscore"]].head())

    # Group by a categorical column (if available)
    non_num = [c for c in df.columns if c not in numeric_cols]
    group_field = None
    for col in non_num:
        if df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped (mean {numeric_field}) by {group_field} (column @id):")
        print(grouped_df.head())
else:
    print("No numeric columns detected for EDA.")

## 5. Visualization
Let us visualize the distribution of the main numeric column, and relationships between two fields where possible.
All visualizations use fields and record sets referenced by their `@id`.

In [ ]:
# Histogram of the main numeric field
if numeric_cols:
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30, alpha=0.7)
    plt.title(f"Histogram of {numeric_field} in {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we have at least two numeric fields, scatter plot
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6,4))
        plt.scatter(df[numeric_cols[0]], df[numeric_cols[1]], alpha=0.5)
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f"Scatter plot: {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.show()

## 6. Conclusion
In this notebook, you used the `mlcroissant` library to discover and analyze the FAIR² Croissant dataset ([schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)), extracted tables and fields by their `@id`, performed exploratory filtering, normalization, grouping, and visualized data distributions.
This workflow can be re-used with any dataset with Croissant schema and record sets. For deeper domain analysis, further cleaning and domain knowledge may be applied.